# PneumoScan — Explainable Pneumonia Screening on Adult Chest X-rays

**Dataset:** RSNA Pneumonia Detection Challenge (adult, radiologist-annotated bounding boxes)
**Task:** binary — NORMAL vs PNEUMONIA (lung opacity)
**Pipeline:** DICOM → CLAHE → augmentation → DenseNet121 → Grad-CAM **scored against the boxes** → TFLite

> **Set the runtime first:** `Runtime → Change runtime type → T4 GPU`.
> **And accept the competition rules once:** https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules
> — click *I Understand and Accept*, or every download returns 403.

A full run is about **60 minutes**. Section 3 self-tests the code in 2 minutes
and section 6 rehearses on a subsample in 20 — both worth doing first.

---
1. GPU check  2. Project code  3. Self-test (optional)  4. Kaggle credentials
5. Download + CLAHE + splits  6. Rehearsal (optional)  7. Train DenseNet121
8. Results  9. Grad-CAM + localisation  10. TFLite  11. Download results

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
!pip install -q pydicom
import tensorflow as tf, keras, pydicom
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| pydicom", pydicom.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus or "NONE  -> Runtime > Change runtime type > T4 GPU, then rerun")

## 2. Get the project code

Pick **one**: 2a if the project is on GitHub, 2b to upload `pneumoscan.zip`.

In [ ]:
# --- 2a. Clone from GitHub -------------------------------------------------
REPO_URL = ""   # e.g. "https://github.com/<you>/pneumoscan.git"

import os, shutil
if REPO_URL:
    shutil.rmtree("/content/project", ignore_errors=True)
    !git clone -q $REPO_URL /content/project
    os.chdir("/content/project")
    print("cloned into", os.getcwd())
    !ls
else:
    print("REPO_URL is empty - use cell 2b instead.")

In [ ]:
# --- 2b. Upload pneumoscan.zip ---------------------------------------------
# Safe to re-run mid-session: this replaces the CODE only. data/, checkpoints/
# and reports/ are left untouched, so a fresh upload never throws away a 3.7 GB
# download or a model you spent 35 minutes training.
import os, glob, zipfile, shutil
from google.colab import files

PROJECT = "/content/project"
KEEP = {"data", "checkpoints", "reports"}

up = files.upload()
name = next(iter(up))
shutil.rmtree("/content/_unzip", ignore_errors=True)
os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall("/content/_unzip")

root = next(iter(glob.glob("/content/_unzip/**/src/train.py", recursive=True)), None)
assert root, "src/train.py not found inside the zip"
src_root = os.path.dirname(os.path.dirname(root))

for item in os.listdir(PROJECT):          # clear old code, keep the expensive stuff
    if item in KEEP:
        continue
    p = os.path.join(PROJECT, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)

for item in os.listdir(src_root):
    shutil.move(os.path.join(src_root, item), PROJECT)

os.chdir(PROJECT)
kept = [d for d in KEEP if os.path.isdir(d)]
print("project at", os.getcwd(), "| preserved:", kept or "nothing yet")
!ls

## 3. Self-test *(optional, ~2 min — worth it)*

Runs the entire pipeline on a small synthetic dataset it generates itself:
DICOM decoding, CLAHE, splits, box rescaling, training, Grad-CAM, the
localisation metric, and the TFLite export. Nothing is downloaded and nothing
outside a temp folder is touched.

If this prints `SELF-TEST PASSED`, the code and this runtime work — anything that
goes wrong later is data or credentials, not the pipeline.

In [ ]:
!python scripts/selftest.py

## 4. Kaggle credentials

Kaggle → avatar → **Settings** → **API** → *Create New Token* downloads `kaggle.json`.

Preferred: add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab **Secrets** (🔑 in the
sidebar) with notebook access enabled. Otherwise the cell falls back to uploading
the file.

In [ ]:
import os, json, pathlib

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("using Colab secrets for", os.environ["KAGGLE_USERNAME"])
except Exception as e:
    print("secrets unavailable (%s) - upload kaggle.json instead" % type(e).__name__)
    from google.colab import files
    up = files.upload()
    creds = json.loads(next(iter(up.values())))
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]

d = pathlib.Path.home() / ".kaggle"; d.mkdir(exist_ok=True)
(d / "kaggle.json").write_text(json.dumps({
    "username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}))
(d / "kaggle.json").chmod(0o600)
print("kaggle.json ready")

## 5. Download, CLAHE, and split  *(~15 min)*

`scripts/prepare_data.py --download`:

* pulls 3.7 GB of DICOMs from Kaggle and unzips them,
* collapses the per-box CSVs into one row per image, carrying the boxes,
* splits 70/15/15, stratified on the label and grouped by patient,
* applies **CLAHE** and writes lossless 224×224 PNGs,
* rescales every radiologist box into the resized frame,
* reads DICOM headers for a cohort table — which is how you *show* the data is adult.

**If this fails with 403**, you have not accepted the competition rules yet:
https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules

In [ ]:
!pip install -q kaggle
!python scripts/prepare_data.py --download

In [ ]:
import pandas as pd
from IPython.display import Image, display

print("Split sizes");     display(pd.read_csv("reports/dataset_summary.csv"))
print("RSNA classes");    display(pd.read_csv("reports/detailed_class_summary.csv", index_col=0))
print("Cohort (adult?)"); display(pd.read_csv("reports/cohort.csv"))
display(Image("reports/clahe_examples.png", width=760))

### Sanity check

Roughly what you should see across all splits combined:

| | images |
|---|---|
| Lung Opacity (label 1) | ~6,000 |
| Normal | ~8,850 |
| No Lung Opacity / Not Normal | ~11,800 |
| **total** | **26,684** |

and about 9,500 boxes. In the cohort table, `pct_under_18` should be near zero
and the median age around 50 — that is your evidence the dataset is adult.

Note the third class: films that are abnormal for some reason *other* than
pneumonia. They are kept as negatives, because a screening tool has to tell
pneumonia from other pathology, not just from healthy lungs. Pass
`--exclude-not-normal` if you want to measure how much easier the task gets
without them.

## 6. Rehearsal on a subsample *(optional, ~20 min — recommended)*

Runs the entire pipeline on 3,000 training images so you find any problem in 20
minutes instead of 2 hours. Validation and test stay full size, so the numbers
are honest, just weaker. Skip to section 7 if you would rather go straight in.

In [ ]:
!python src/train.py --tag rehearsal --subsample-train 3000 --head-epochs 1 --finetune-epochs 2
!python scripts/make_gradcam_figures.py --tag rehearsal --loc-max 300

## 7. Train DenseNet121  *(~35 min)*

Two phases: 3 warm-up epochs with the backbone frozen, then fine-tuning that
early-stops on validation AUROC. `--finetune-epochs 20` raises the ceiling if you
want to give it more room; early stopping decides when to actually halt.

In [ ]:
!python src/train.py

## 7. Results

Reads what training wrote to `reports/densenet121/metrics.json`. Model selection
and the decision threshold both come from **validation**; the test set is only
ever reported on.

In [ ]:
import json, pandas as pd
from pathlib import Path

runs = [f for f in sorted(Path("reports").glob("*/metrics.json"))
        if "rehearsal" not in f.parent.name]
assert runs, "no completed run found - train the model first (section 7)"

rows = []
for f in runs:
    m = json.loads(f.read_text())
    lo, hi = m["test_auroc_95ci"]
    rows.append({"run": f.parent.name,
                 "val AUROC": round(m["val"]["auroc"], 4),
                 "test AUROC": round(m["test"]["auroc"], 4),
                 "95% CI": "%.3f-%.3f" % (lo, hi),
                 "accuracy": round(m["test"]["accuracy"], 4),
                 "sensitivity": round(m["test"]["sensitivity_recall"], 4),
                 "specificity": round(m["test"]["specificity"], 4),
                 "F1": round(m["test"]["f1"], 4),
                 "threshold": round(m["test"]["threshold"], 3),
                 "min": m.get("train_minutes")})
results = pd.DataFrame(rows)
results.to_csv("reports/results.csv", index=False)
BEST = results.iloc[0]["run"]
results

In [ ]:
from IPython.display import Image, display
for f in ("training_curves.png", "test_curves.png", "test_confusion.png"):
    display(Image(f"reports/{BEST}/{f}", width=820))

## 8. Grad-CAM — and the number that makes it evidence

RSNA gives radiologist-drawn boxes, so the heatmaps can be *scored* rather than
admired:

* **pointing game** — does the hottest pixel land inside a box?
* **energy pointing game** — what share of the heatmap's mass is inside the boxes?
* **IoU@0.5** — overlap after thresholding at half the peak.

Each is reported against a **chance baseline** (the boxes' own share of the
image). A pointing-game score means nothing without it — that ratio, the *lift*,
is the number to quote.

In [ ]:
# Section 8 (Results) sets BEST. If you skipped it, fall back to whatever was trained,
# so this cell never dies with a NameError after a long training run.
from pathlib import Path
if "BEST" not in globals():
    trained = sorted(c.stem for c in Path("checkpoints").glob("*.keras")
                     if "rehearsal" not in c.stem)
    assert trained, "no trained model found in checkpoints/"
    BEST = trained[0]
print("explaining:", BEST)

In [ ]:
!python scripts/make_gradcam_figures.py

In [ ]:
import pandas as pd
from IPython.display import Image, display

loc = pd.read_csv(f"reports/{BEST}/localization.csv")
display(loc[["explainer", "n", "pointing_game", "pointing_game_chance",
             "pointing_game_lift", "energy_pointing_game"]])

for f in ("gradcam_pneumonia.png", "gradcam_errors.png",
          "gradcam_vs_pp.png", "gradcam_normal.png"):
    try:
        display(Image(f"reports/{BEST}/{f}", width=900))
    except Exception:
        print("missing:", f)

## 9. Export to TensorFlow Lite

Writes a float32 and a dynamic-range int8 model, and refuses to ship the float
one if it drifts from Keras by more than 1e-3 on real test images.

In [ ]:
!python src/export_tflite.py

## 10. Download the results

In [ ]:
!zip -qr /content/results.zip reports checkpoints -x "*.npz"
from google.colab import files
files.download("/content/results.zip")

In [ ]:
# Safer on a flaky connection: keep everything on Drive as you go.
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/pneumoscan" && cp -r reports checkpoints "/content/drive/MyDrive/pneumoscan/"